## Part 1: Setting Up

When we use regular software on a computer, we usually follow 3 steps:

1. Install the software.
2. Open it.
3. Sign in with a username and password (if required, like in Outlook or Microsoft Office).

In Python, we follow a very similar process:

1. Instead of installing software → we install a **package**.
2. Instead of opening a program → we **import** from the package.
3. Instead of signing in with username and password → we use an **API key**.

If you've ever installed and used software before - you already understand the idea: we're just doing it in code.

Let's get started - you're only **3 lines of code** away from talking to an AI model!

### Step 1: Install

Run the next cell to install everything we need (wait until it is done).

What does this command does?

- `pip` - a command for installing packages (we can't use a mouse here, so we type commands 🙂).
- `install` - tells pip what to do (install the packages).
- `langchain` - the first package we install. It helps us build applications with AI models.
- `langchain-openai` – the second package we install. It lets us use OpenAI models through LangChain.

In [ ]:
!pip install langchain langchain-openai


[notice] A new release of pip available: 22.2.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Import

Now we can use what we installed using `import`.

Think of it like taking a tool from a toolbox and run the next cell. It says:
"From the toolbox `langchain_openai`, bring the tool `ChatOpenAI`"

In [ ]:
from langchain_openai import ChatOpenAI

### Step 3: Authenticate

Now we create the model and give it a name: `llm` (short for Large Language Model. Large Language Model is a type of AI that can understand and generate text).

We could call it anything (like `mitzi` or `david`), but we choose a meaningful name.

We also:
- choose a model (`gpt-4o-mini` - a simple and low-cost option).
- provide an API key - like a password that identifies us.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key="WRITE THE KEY HERE")

That's it! just 3 lines and everything is ready!

Ready to talk to an AI model?

Run the next cell and see the response (it will be presented nicely using pretty print):

In [ ]:
response = llm.invoke("Introduce yourself. Which model are you?")
response.pretty_print()

I am based on OpenAI's GPT-3 model, a sophisticated language model designed to understand and generate human-like text. My purpose is to assist with a variety of tasks, including providing information, answering questions, and engaging in conversation. How can I help you today?


Try it yourself.

Use the same code, but change the prompt text.

For example:
- "Tell me a joke"
- "Explain what AI is in simple words"

In [ ]:
response = llm.invoke("Write one sentence about why tools are useful for LLMs.")
response.pretty_print()

================================== Ai Message ==================================

I am ChatGPT, based on the GPT-4 architecture developed by OpenAI. I'm designed to assist with a wide range of questions and tasks, providing information, answering queries, and engaging in conversation. How can I help you today?


Check point - is all good? Let's wait for the others...

## Part 2: Creating a Python tool

In this part, we will create a Python tool (in Python terms - a `Python function`).

Our made-up tool, called `mojifuzzle`, will create emojis for us:
hearts ❤️ or stars ⭐ - as many as we want.

Run the code below to create the tool.

Even if you don't know Python, try to guess what it does 🙂 - Here are a few helpful terms:

- `def` - short for "define".
- `str` - short for "string" (text).
- `int` - short for "integer" (a whole number).
- `elif` - short for "else if".
- `*` - multiplication (e.g. 2 * 3 = 6).

### 😊 Don’t worry

You don't need to fully understand the code (functions are one of the more challenging topics, and that's not our goal here).
For now, it’s enough to get a general idea of what the code does.

In [ ]:
def mojifuzzle(emoji_name: str, count: int) -> str:
    if emoji_name == "heart":
        return "❤️" * count
    elif emoji_name == "star":
        return "⭐" * count
    else:
        return "🤔"

Give it a try!

Run the next few cells and see our tool in action:

In [ ]:
mojifuzzle("heart", 10)

'❤️❤️❤️❤️❤️❤️❤️❤️❤️❤️'

In [ ]:
mojifuzzle("star", 2)

'⭐⭐'

In [ ]:
mojifuzzle("moon", 2)

'🤔'

Check point - is all good? Let's wait for the others...

## Part 3: From a Python Tool to an AI Tool

What do you think will happen if we ask the model:

`please mojifuzzle 3 times with a heart`?

Run the next cell 3-4 times and see what happens.

In [ ]:
response = llm.invoke("Please mojifuzzle 3 times with a heart.")
response.pretty_print()

Sure! Here’s "mojifuzzle" with a heart emoji three times:

💖 mojifuzzle 💖 mojifuzzle 💖 mojifuzzle 💖


As you probably noticed, the model improvised: It tried to respond based only on the text prompt and its own reasoning.  
It did **not** use our mojifuzzle.

So what can we do?

We can introduce our tool to the AI model.

In code, this is very simple:

![Diagram](images/functionToTool.png)

The `@tool` line tells LangChain:
"This function is a tool the AI can use".

LangChain then converts the function  
into structured text the model can understand.

In [ ]:
from langchain.tools import tool

@tool
def mojifuzzle(emoji_name: str, count: int) -> str:
    """Create a string of emojis using 'heart' or 'star'."""
    if emoji_name == "heart":
        return "❤️" * count
    elif emoji_name == "star":
        return "⭐" * count
    else:
        return "🤔"

#### Optional: Run the following cell to see the schema created yourself:

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool
import json

print(json.dumps(convert_to_openai_tool(mojifuzzle), indent=2))

{
  "type": "function",
  "function": {
    "name": "mojifuzzle",
    "description": "Create emojis using 'heart' or 'star'.",
    "parameters": {
      "properties": {
        "emoji_name": {
          "type": "string"
        },
        "count": {
          "type": "integer"
        }
      },
      "required": [
        "emoji_name",
        "count"
      ],
      "type": "object"
    }
  }
}


Check point - is all good? Let's wait for the others...

### Step 4: Now the AI Can Use the Tool

So, we have a model, and we have a tool the AI can use — let’s connect them!

In more technical language, you might hear the term "bind"  
(it sounds a bit scary, but it simply means connecting the tool to the model 🙂).

And that’s exactly what we’ll do.

We will bind the tool to the model and create an improved version called `llm_with_tools`.

Run the next cell to do exactly that:

In [18]:
llm_with_tools = llm.bind_tools([mojifuzzle])

Now let's send the same request as before:

In [ ]:
response = llm_with_tools.invoke("Please mojifuzzle 3 times with a heart.")
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  mojifuzzle (call_ZTYX7DsmSyXhg4NUJLS90C4X)
 Call ID: call_ZTYX7DsmSyXhg4NUJLS90C4X
  Args:
    emoji_name: heart
    count: 3


Let’s examine the output.

This time, instead of just improvising, the model says:

"I’ve seen a tool that can do this. It’s called `mojifuzzle`.  
If you use it with:

- `emoji_name="heart"`  
- `count=3`

you’ll get exactly what you want."

Not that impressive, right?

Well… let’s see what happens when things get a bit harder:

In [ ]:
response = llm_with_tools.invoke("Please mojifuzzle the symbol of love for the number of sides in a triangle. Make sure to provide only a single output.")
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  mojifuzzle (call_ZmfTwOGrEW2u6cKUzQz3my18)
 Call ID: call_ZmfTwOGrEW2u6cKUzQz3my18
  Args:
    emoji_name: heart
    count: 3


## Summary so far

### Part 1: Setting Up
- Installed the required packages  
- Imported the model  
- Authenticated using an API key  
- Sent our first prompt to the model  

### Part 2: Creating a Python Tool
- Created a Python function (`mojifuzzle`)  
- Understood (at a high level) what the function does  
- Used it directly in Python  

### Part 3: From a Python Tool to an AI Tool
- Saw that the model cannot use our function by default  
- Introduced the function as a tool using `@tool`  
- Learned that the model sees the tool as structured textual information  

### Step 4: Now the AI Can Use the Tool
- Bound the tool to the model  
- Observed how the model chooses to use the tool instead of improvising

Check point - is all good? Let's wait for the others...

### Part 5: Closing the Loop - Letting AI Actually Use the Tool

So far, we’ve seen how the model can *decide* to use a tool.

But something is still missing:

👉 Who actually runs the tool?

To make this work end-to-end, we need a small system  
that can:
- listen to the model  
- run the tool  
- return the result  

This is often called an **agent**: an agent combines an LLM with tools and logic to take actions.

You can think of it like this:

- The **LLM** decides what to do  
- The **agent** actually does it

Instead of building everything from scratch,  
we’ll use an existing agent: **Claude Desktop (MCP client)**.

---

### Step 1: Create an MCP server

We will wrap our tool in a small server:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Mojifuzzle server")

@mcp.tool()
def mojifuzzle(emoji_name: str, count: int) -> str:
    """
    Create a string of emojis.
    Use 'heart' or 'star'.
    """
    if emoji_name == "heart":
        return "❤️" * count
    elif emoji_name == "star":
        return "⭐" * count
    else:
        return "🤔"

if __name__ == "__main__":
    mcp.run()
```

### Step 2: Run it as a separate program

Place this code in a Python file and run it outside the notebook.

### Step 3: Connect from an MCP client

Now we can connect to it using an MCP client like Claude Desktop.

In the live demo, the client will:

1. Discover the server.
2. See that mojifuzzle is available as a tool.
3. Decide when to call it.
4. Run the tool and get the result.
5. Continue the conversation.